# FaceSense AI Dataset Exploration

This notebook loads the processed UTKFace labels, explores age and gender distributions, and displays a random sample of processed faces.

## Load labels

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RANDOM_STATE = 42
NOTEBOOK_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = NOTEBOOK_ROOT / "data" / "processed"
LABELS_PATH = PROCESSED_DIR / "labels.csv"
REQUIRED_COLUMNS = {"filename", "age", "gender", "split"}
GENDER_LABELS = {0: "Male", 1: "Female"}

if not LABELS_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {LABELS_PATH}. Run `python data/prepare_dataset.py` "
        "from the project root first."
    )

labels = pd.read_csv(LABELS_PATH)
missing_columns = REQUIRED_COLUMNS - set(labels.columns)
if missing_columns:
    raise ValueError(f"labels.csv is missing columns: {sorted(missing_columns)}")

labels["gender_label"] = labels["gender"].map(GENDER_LABELS)
if labels["gender_label"].isna().any():
    raise ValueError("gender must contain only 0 (male) or 1 (female)")

print(f"Loaded {len(labels):,} labeled images from {LABELS_PATH}")
labels.head()

## Age histogram

In [ ]:
assert not labels[["age", "gender"]].isnull().any().any(), (
    "Labels must not contain null age or gender values"
)
assert labels["age"].between(1, 116).all(), "Ages must be between 1 and 116"
assert labels["gender"].isin([0, 1]).all(), "Gender values must be only 0 or 1"

print("Label validation passed: no nulls, valid ages, and valid gender values.")

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(labels["age"], bins=20, color="steelblue", edgecolor="white")
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Number of images")
plt.grid(axis="y", alpha=0.25)
plt.show()

## Gender distribution

In [ ]:
gender_counts = labels["gender_label"].value_counts().reindex(["Male", "Female"], fill_value=0)

plt.figure(figsize=(7, 5))
bars = plt.bar(gender_counts.index, gender_counts.values, color=["#4c78a8", "#f58518"])
plt.title("Gender Distribution")
plt.xlabel("Gender")
plt.ylabel("Number of images")
for bar, count in zip(bars, gender_counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2, count, f"{count:,}", ha="center", va="bottom")
plt.show()

## Age distribution by gender

In [ ]:
age_by_gender = [labels.loc[labels["gender_label"] == gender, "age"] for gender in ["Male", "Female"]]

plt.figure(figsize=(8, 5))
plt.boxplot(age_by_gender, labels=["Male", "Female"], showfliers=False)
plt.title("Age Distribution by Gender")
plt.xlabel("Gender")
plt.ylabel("Age")
plt.grid(axis="y", alpha=0.25)
plt.show()

## Random sample of 16 faces

In [ ]:
sample = labels.sample(n=min(16, len(labels)), random_state=RANDOM_STATE)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
axes = axes.ravel()

for axis, (_, row) in zip(axes, sample.iterrows()):
    image_path = PROCESSED_DIR / row["split"] / row["filename"]
    axis.axis("off")
    if image_path.exists():
        axis.imshow(plt.imread(image_path))
        axis.set_title(
            f"Age: {row['age']} | {row['gender_label']}\n{row['split']}",
            fontsize=9,
        )
    else:
        axis.set_title(f"Missing: {row['filename']}", fontsize=9, color="red")

for axis in axes[len(sample):]:
    axis.axis("off")
fig.suptitle("Random Processed Face Samples", fontsize=16)
plt.tight_layout()
plt.show()

## Class imbalance summary

In [ ]:
total = gender_counts.sum()
percentages = gender_counts / total * 100
majority_count = gender_counts.max()
minority_count = gender_counts.min()
majority_gender = gender_counts.idxmax()
minority_gender = gender_counts.idxmin()
ratio = majority_count / minority_count if minority_count else float("inf")

print("Gender class balance:")
for gender in gender_counts.index:
    print(f"- {gender}: {gender_counts[gender]:,} ({percentages[gender]:.2f}%)")
print(
    f"Majority class: {majority_gender}; minority class: {minority_gender}; "
    f"majority/minority ratio: {ratio:.2f}:1"
)
if ratio <= 1.10:
    print("The gender classes are approximately balanced.")
else:
    print("The gender classes show noticeable imbalance; consider stratification or class weights.")